# Convert NDJSON to CSV with chDB

In-process ClickHouse SQL in Python. Run `./generate.sh` first to create `data/events.ndjson`.

In [ ]:
import chdb

A naive `SELECT *` splits the nested `device` object (inferred as a Tuple) across extra columns without extra headers, so the CSV header count and data column count no longer match. Project the nested fields out explicitly into flat columns instead, and serialise the `tags` array as a JSON string so it stays in one cell.

In [ ]:
sql = """
SELECT
  event_id,
  ts,
  country,
  action,
  amount,
  device.os          AS device_os,
  device.app_version AS device_version,
  toJSONString(tags) AS tags
FROM file('data/events.ndjson')
INTO OUTFILE 'data/events_flat_chdb.csv' TRUNCATE FORMAT CSVWithNames
"""
chdb.query(sql)
print("wrote data/events_flat_chdb.csv")

Read the result back as a DataFrame to confirm it is clean tabular data.

In [ ]:
df = chdb.query(
    "SELECT * FROM file('data/events_flat_chdb.csv') ORDER BY event_id LIMIT 3",
    "DataFrame",
)
print(df)